# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import json
from datetime import datetime

NOTES_DIR = "notes"
os.makedirs(NOTES_DIR, exist_ok=True)



In [ ]:
load_dotenv(override=True)

PROVIDERS = {
    "Ollama (local)": {
        "client": OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollama"),
        "model": "llama3.2",
    },
    "Groq": {
        "client": OpenAI(base_url=os.getenv("GROQ_BASE_URL"), api_key=os.getenv("GROQ_API_KEY")),
        "model": "llama-3.1-8b-instant",   # or "llama-3.3-70b-versatile"
    },
    "Gemini": {
        "client": OpenAI(base_url=os.getenv("GOOGLE_BASE_URL"), api_key=os.getenv("GOOGLE_API_KEY")),
        "model": "gemini-3.1-flash-lite",
    },
}

In [ ]:
EXPERTISE_PROMPTS = {
    "Python Developer": "You are a senior Python developer. Explain code line-by-line, covering behavior, purpose, and edge cases.",
    "JavaScript Developer": "You are a senior JavaScript/frontend developer. Explain code clearly, including async behavior, browser APIs, and gotchas.",
    "SQL / Data Engineer": "You are a senior data engineer. Explain queries or data code in terms of performance, correctness, and edge cases.",
    "Beginner-Friendly Tutor": "You are a patient coding tutor explaining to someone new to programming. Avoid jargon, use analogies.",
    "System Design / Architecture": "You are a staff engineer reviewing code for architecture, scalability, and maintainability concerns.",
}

In [ ]:
def explain_code_stream(code: str, provider_name: str, expertise: str):
    provider = PROVIDERS[provider_name]
    client = provider["client"]
    model = provider["model"]

    messages = [
        {"role": "system", "content": EXPERTISE_PROMPTS[expertise]},
        {"role": "user", "content": f"Explain the following code in detail:\n\n{code}\n\nInclude how it works and why each part is used."},
    ]

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )

    partial = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        partial += delta
        yield partial   # yielding the growing string is what makes Gradio "type" it out live

In [ ]:
def save_full_response(history):
    if not history:
        return None
    last_question, last_answer = history[-1]          # gr.Chatbot stores (user, bot) tuples
    safe_name = "".join(c if c.isalnum() else "_" for c in last_question[:40])
    filepath = os.path.join(NOTES_DIR, f"{safe_name}.md")
    with open(filepath, "w") as f:
        f.write(f"# {last_question}\n\n{last_answer}")   # full verbatim answer, no summarizing
    return filepath

In [ ]:
def chat(message, history, provider_name, expertise):
    provider = PROVIDERS[provider_name]
    client, model = provider["client"], provider["model"]

    messages = [{"role": "system", "content": EXPERTISE_PROMPTS[expertise] +
                 " If the user's message is a technical question worth remembering, call save_explanation."}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    # Step 1: non-streaming call to check for tool use
    check = client.chat.completions.create(model=model, messages=messages, tools=tools)
    msg = check.choices[0].message
    status = ""

    

    # Step 2: stream the real answer
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    partial = status
    for chunk in stream:
        partial += chunk.choices[0].delta.content or ""
        yield partial    
        provider = PROVIDERS[provider_name]
    client, model = provider["client"], provider["model"]

    messages = [{"role": "system", "content": EXPERTISE_PROMPTS[expertise]}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    partial = ""
    for chunk in stream:
        partial += chunk.choices[0].delta.content or ""
        yield partial



In [ ]:
with gr.Blocks() as demo:
    chat_ui = gr.ChatInterface(
        fn=chat,
        additional_inputs=[
            gr.Dropdown(list(PROVIDERS.keys()), value="Ollama (local)", label="Provider"),
            gr.Dropdown(list(EXPERTISE_PROMPTS.keys()), value="Python Developer", label="Expertise"),
        ],
    )
    save_btn = gr.Button("💾 Save last response to file")
    file_out = gr.File(label="Download")
    save_btn.click(fn=save_full_response, inputs=[chat_ui.chatbot], outputs=[file_out])

demo.launch(inbrowser=True, auth=("sheri", "abc.123"))